# Exercice 3 : Inventorier l'environnement (ML-BOM) et trouver ses vulnérabilités

> **Objectif :** produire l'inventaire CycloneDX (ML-BOM) de l'environnement du cours, avec notre modèle et ses données,
> puis le confronter aux bases de vulnérabilités avec `pip-audit` pour faire ressortir de vraies CVE.

**Le scénario.** Un incident comme torchtriton (déc. 2022) ou Ultralytics (déc. 2024) est annoncé. La première question du RSSI : « Sommes-nous touchés ? Où ? »
Sans inventaire, personne ne peut répondre. Avec un ML-BOM, c'est une recherche dans un fichier.

**Le plan.**
1. Générer l'inventaire (SBOM) de tous les paquets Python de l'environnement.
2. Y ajouter les composants IA : le jeu de données et le modèle (avec leurs empreintes). On obtient un ML-BOM.
3. Lancer `pip-audit` sur l'environnement du cours.
4. Lancer `pip-audit` sur les dépendances figées d'un projet ML typique de 2024.
5. Vérifier qu'un nom de paquet existe vraiment (slopsquatting).

## Étape 0 : exécutez cette cellule en premier
Elle liste les dépendances. Sur Colab (notebook ouvert depuis Drive), elle télécharge `helper.py` et les données. Elle doit afficher **Prêt : True**.

In [1]:
# Dépendances : cyclonedx-bom pip-audit pandas requests
# Récupère helper.py et les données depuis GitHub s'ils ne sont pas à côté du notebook
# (cas d'un notebook ouvert depuis Google Drive). En local, cette cellule ne fait rien.
import os, sys
if not os.path.exists('helper.py'):
    !test -d /content/cours || git clone -q --depth 1 https://github.com/elyesmanai/8IAR805---Cybersecurity-AI.git /content/cours
    %cd "/content/cours/Chapitre 6 - Chaîne d'approvisionnement des modèles"
    !pip -q install cyclonedx-bom pip-audit
print('Prêt :', os.path.exists('helper.py'))

Prêt : True


## Étape 1 : L'inventaire logiciel de l'environnement
**Ce qu'on fait :** `cyclonedx-py environment` lit les métadonnées de chaque paquet installé et écrit un fichier CycloneDX 1.6 (JSON). Cela prend ≈ 30 secondes.

In [2]:
import json, pandas as pd
from helper import sbom_environnement
bom = sbom_environnement('data/sbom_environnement.json')
print(f"Format : CycloneDX {bom['specVersion']} | {len(bom['components'])} composants")
pd.DataFrame([{'nom': c['name'], 'version': c.get('version'), 'purl': c.get('purl'), 'type': c['type']}
              for c in bom['components'] if c['name'].lower() in ('scikit-learn', 'numpy', 'pandas', 'onnxruntime', 'transformers', 'huggingface-hub')])

Format : CycloneDX 1.6 | 268 composants


,nom,version,purl,type
0,numpy,1.26.4,pkg:pypi/numpy@1.26.4,library
1,onnxruntime,1.31.1,pkg:pypi/onnxruntime@1.31.1,library
2,pandas,2.1.4+dfsg,pkg:pypi/pandas@2.1.4%2Bdfsg,library
3,scikit-learn,1.9.0,pkg:pypi/scikit-learn@1.9.0,library
4,transformers,5.17.0,pkg:pypi/transformers@5.17.0,library


**👀 Ce qu'on doit voir :** environ 250 composants, tous de type `library`. Chacun a un **purl** (*package URL*, ex. `pkg:pypi/scikit-learn@1.9.0`) : c'est l'identifiant que les bases de vulnérabilités utilisent.

Il manque l'essentiel pour un système d'IA : **les données et le modèle**. Un SBOM classique ne les voit pas.

## Étape 2 : Ajouter le modèle et ses données (ML-BOM)
**Ce qu'on fait :** CycloneDX (depuis la version 1.5, juin 2023) a deux types de composants pour l'IA : `data` et `machine-learning-model`.
On ajoute le jeu de données du Ch. 1 et la forêt aléatoire ONNX de l'exercice 2, avec leurs empreintes SHA-256 et une fiche de modèle (`modelCard`).

In [3]:
from helper import composant_donnees, composant_modele, ajouter_au_bom, valider_bom, MODELES, DATA

donnees = composant_donnees('dataset_malware', f'{DATA}/dataset_malware.csv',
                            "19 611 exécutables Windows (77 champs de l'en-tête PE), étiquetés maliciel / bénin", 'Ch. 1 du cours 8IAR805')
modele = composant_modele('detecteur-maliciels-foret', f'{MODELES}/foret.onnx', donnees['bom-ref'],
                          tache='classification', famille='random forest', metriques={'accuracy': 0.9898},
                          dependances=['onnxruntime', 'onnx', 'scikit-learn', 'skl2onnx'])
ml_bom = ajouter_au_bom(bom, [donnees, modele])
json.dump(ml_bom, open('data/ml-bom.json', 'w'), indent=2)
print('Validation contre le schéma CycloneDX 1.6 :', valider_bom('data/ml-bom.json'))
print(json.dumps(modele, indent=2, ensure_ascii=False))

Validation contre le schéma CycloneDX 1.6 : valide
{
  "type": "machine-learning-model",
  "bom-ref": "modele:detecteur-maliciels-foret",
  "name": "detecteur-maliciels-foret",
  "version": "1.0.0",
  "hashes": [
    {
      "alg": "SHA-256",
      "content": "4c9132f72f6c3860522eb2a9fa0c6e81d381a4d4ca3f04819ee4dc7b5596b75a"
    }
  ],
  "properties": [
    {
      "name": "format",
      "value": "onnx"
    }
  ],
  "modelCard": {
    "modelParameters": {
      "approach": {
        "type": "supervised"
      },
      "task": "classification",
      "architectureFamily": "random forest",
      "datasets": [
        {
          "ref": "donnees:dataset_malware"
        }
      ]
    },
    "quantitativeAnalysis": {
      "performanceMetrics": [
        {
          "type": "accuracy",
          "value": "0.9898"
        }
      ]
    }
  }
}


In [4]:
# Les dépendances déclarées du modèle : c'est ce qui permet de répondre à « sommes-nous touchés ? »
[d for d in ml_bom['dependencies'] if d['ref'] == modele['bom-ref']]

[{'ref': 'modele:detecteur-maliciels-foret',
  'dependsOn': ['onnxruntime==1.31.1',
   'onnx==1.23.2',
   'scikit-learn==1.9.0',
   'skl2onnx==1.20.0']}]

**👀 Ce qu'on doit voir :** le fichier est **valide** selon le schéma officiel. Le modèle a son empreinte, sa tâche, sa famille, sa métrique, un lien vers son jeu de données et la liste des bibliothèques dont il dépend.
C'est ce que l'OWASP (LLM03:2025) recommande : un inventaire **signé** et à jour, y compris les modèles et les données.

## Étape 3 : `pip-audit` sur l'environnement du cours
**Ce qu'on fait :** `pip-audit` compare chaque paquet installé à la base de vulnérabilités de PyPI (alimentée par OSV). Cela prend ≈ 20 secondes.

In [5]:
from helper import pip_audit
vulns_env, _ = pip_audit(['-l'])
nb_paquets = sum(c['type'] == 'library' for c in ml_bom['components'])
print(f"{vulns_env['paquet'].nunique()} paquets vulnérables sur {nb_paquets}, {len(vulns_env)} vulnérabilités au total")
vulns_env.groupby('paquet').agg(version=('version', 'first'), nb=('id', 'count'), exemple=('CVE', 'first')).sort_values('nb', ascending=False).head(10)

25 paquets vulnérables sur 268, 181 vulnérabilités au total


,version,nb,exemple
paquet,,,
pillow,10.2.0,48,CVE-2024-28219
pypdf,6.16.2,26,CVE-2026-102994
gitpython,3.1.57,19,CVE-2026-87819
pyjwt,2.7.0,14,CVE-2026-32597
cryptography,41.0.7,9,CVE-2023-50782
tornado,6.5.7,9,CVE-2026-82397
yt-dlp,2024.4.9,8,CVE-2024-38519
urllib3,2.0.7,8,CVE-2024-37891
pip,24.0,6,CVE-2025-8869


**👀 Ce qu'on doit voir :** une vingtaine de paquets vulnérables et plus d'une centaine de vraies CVE (`pillow`, `cryptography`, `urllib3`, `jinja2`, `pip`, …).
Rien d'exotique : ce sont des dépendances **transitives** que personne n'a installées volontairement. Le chiffre exact dépend de votre machine.

**❓ Question pour la classe :** faut-il tout mettre à jour tout de suite ? *Réponse : non, on priorise. Une CVE dans un paquet exposé au réseau ou qui charge des fichiers non fiables passe avant une CVE dans un outil de développement.*

## Étape 4 : Un projet ML typique de 2024
**Ce qu'on fait :** on audite un `requirements.txt` figé comme on en trouve partout : PyTorch, Transformers, Ray, Ultralytics, scikit-learn.
On n'installe rien : `pip-audit` lit seulement les noms et les versions.

In [6]:
projet_2024 = '''torch==2.5.1
transformers==4.36.0
ray==2.6.0
ultralytics==8.3.41
scikit-learn==1.4.0
'''
open('data/requirements_projet_2024.txt', 'w').write(projet_2024)
vulns_2024, ignores = pip_audit(['-r', 'data/requirements_projet_2024.txt', '--no-deps', '--disable-pip'])
print(vulns_2024.groupby('paquet').size().rename('nb de vulnérabilités').to_string())
ignores

paquet
ray             11
scikit-learn     1
torch           19
transformers    27


,paquet,raison
0,ultralytics,Dependency not found on PyPI and could not be ...


In [7]:
# Celles qui concernent directement ce chapitre
a_voir = ['CVE-2025-32434', 'CVE-2023-48022', 'CVE-2024-5206', 'CVE-2024-3568']
vulns_2024[vulns_2024['CVE'].isin(a_voir)][['paquet', 'version', 'CVE', 'corrigé dans', 'description']]

,paquet,version,CVE,corrigé dans,description
2,torch,2.5.1,CVE-2025-32434,2.6.0,# Description I found a Remote Command Executi...
20,transformers,4.36.0,CVE-2024-3568,4.38.0,The huggingface/transformers library is vulner...
50,ray,2.6.0,CVE-2023-48022,(aucun correctif),Anyscale Ray allows a remote attacker to execu...
57,scikit-learn,1.4.0,CVE-2024-5206,1.5.0,A sensitive data leakage vulnerability was ide...


**👀 Ce qu'on doit voir :**
- **`torch` CVE-2025-32434** : `torch.load(..., weights_only=True)`, la protection recommandée contre les pickles piégés, pouvait quand même exécuter du code avant la version 2.6.0 ;
- **`ray` CVE-2023-48022** : l'API de soumission de tâches sans authentification exploitée par la campagne **ShadowRay** (2023-2024). Pas de correctif : l'éditeur la conteste et la considère comme une fonctionnalité ;
- **`scikit-learn` CVE-2024-5206** : le `TfidfVectorizer` gardait en mémoire des mots qui pouvaient contenir des données sensibles (corrigé en 1.5.0) ;
- **`transformers` CVE-2024-3568** : désérialisation de données non fiables (exécution de code) ;
- **`ultralytics` 8.3.41** n'est **pas trouvée** sur PyPI : c'est la version piégée (mineur de cryptomonnaie) retirée en décembre 2024. Un audit qui « ignore » un paquet est une alerte.

## Étape 5 : Ce paquet existe-t-il vraiment ? (slopsquatting)
**Ce qu'on fait :** un assistant de code propose parfois des paquets qui **n'existent pas** : 5,2 % des paquets proposés par les modèles commerciaux et 21,7 % par les modèles ouverts, 205 474 noms inventés distincts (Spracklen et al., USENIX Security 2025).
Un attaquant peut enregistrer ce nom. Avant d'installer, on vérifie sur PyPI : le paquet existe-t-il, depuis quand, combien de versions ?

In [8]:
import requests

def verifier_pypi(nom):
    try:
        r = requests.get(f'https://pypi.org/pypi/{nom}/json', timeout=10)
    except requests.RequestException as e:
        return {'paquet': nom, 'sur PyPI': f'[REPLI] réseau indisponible ({type(e).__name__})'}
    if r.status_code != 200:
        return {'paquet': nom, 'sur PyPI': 'ABSENT (aucune version publiée)'}
    infos = r.json()
    dates = sorted(f['upload_time'] for fichiers in infos['releases'].values() for f in fichiers)
    return {'paquet': nom, 'sur PyPI': 'oui', 'première version': dates[0][:10] if dates else '?',
            'nb de versions': len(infos['releases']), 'résumé': infos['info']['summary'][:60]}

# Liste proposée par un assistant IA pour « télécharger un modèle Hugging Face et détecter des objets »
pd.DataFrame([verifier_pypi(n) for n in ['huggingface-hub', 'huggingface-cli', 'ultralytics', 'torchtriton', 'scikit-learn']])

,paquet,sur PyPI,première version,nb de versions,résumé
0,huggingface-hub,oui,2020-12-23,338.0,"Client library to download and publish models,..."
1,huggingface-cli,ABSENT (aucune version publiée),NaN,NaN,NaN
2,ultralytics,oui,2022-04-04,874.0,"Ultralytics YOLO 🚀 for SOTA object detection, ..."
3,torchtriton,ABSENT (aucune version publiée),NaN,NaN,NaN
4,scikit-learn,oui,2011-09-22,76.0,A set of python modules for machine learning a...


**👀 Ce qu'on doit voir :** `huggingface-cli` est **absent**. C'est pourtant le nom que des LLM inventaient régulièrement en 2024 (le vrai paquet est `huggingface-hub`).
Bar Lanyado (Lasso Security) l'avait enregistré, vide, pour tester : plus de 15 000 téléchargements en trois mois, et un dépôt d'Alibaba le recommandait dans son README.
`torchtriton` est aussi absent aujourd'hui ; en décembre 2022, il était présent… et piégé.

Un nom **absent** dans un `requirements.txt` est donc une alerte : soit une faute de frappe, soit une hallucination, et dans les deux cas un attaquant peut réserver ce nom s'il est libre.

## Ce qu'il faut retenir
- Un **ML-BOM** (CycloneDX ≥ 1.5) inventorie les bibliothèques **et** les modèles et les données, avec leurs empreintes. C'est ce qui permet de répondre en minutes à « sommes-nous touchés ? ».
- `pip-audit` transforme l'inventaire en liste de vraies CVE. Les vulnérabilités sont surtout dans les dépendances **transitives**.
- Les bibliothèques d'IA ont leurs propres CVE critiques : chargement de modèles (`torch`, `transformers`), infrastructure (`ray`).
- Un paquet **introuvable** ou très récent dans une liste de dépendances est une alerte (Ultralytics retiré, noms hallucinés).

**Exercice suivant :** l'inventaire dit **quels** composants on utilise, pas **ce qu'ils font**. Un modèle peut avoir une empreinte valide, une signature valide… et une porte dérobée.